In [6]:
import pandas as pd

# Leer archivos
df_obd = pd.read_csv(r"14_05_26\obd_speed.csv")
df_imu = pd.read_csv(r"14_05_26\csv_output\prueba_2.csv")

# Convertir timestamps
df_obd["timestamp"] = pd.to_datetime(df_obd["timestamp"])
df_imu["timestamp"] = pd.to_datetime(df_imu["timestamp"])

In [7]:
import pandas as pd

# Ejemplo: df_obd es tu dataframe OBD
# Debe tener columnas:
# time_unix_ms
# timestamp

imu_ref_ms = 1778777340292
obd_ref_ms = 1778759349536

offset_ms = imu_ref_ms - obd_ref_ms

print("Offset ms:", offset_ms)
print("Offset s :", offset_ms / 1000)

# Aplicar offset al tiempo Unix del OBD
df_obd["time_unix_ms_corr"] = df_obd["time_unix_ms"] + offset_ms

# Convertir a timestamp corregido
df_obd["timestamp_corr"] = pd.to_datetime(
    df_obd["time_unix_ms_corr"],
    unit="ms"
)

# Revisar
df_obd[["time_unix_ms", "timestamp", "time_unix_ms_corr"]].head()

Offset ms: 17990756
Offset s : 17990.756


,time_unix_ms,timestamp,time_unix_ms_corr
0,1778759273072,2026-05-14 11:47:53.072,1778777263828
1,1778759274488,2026-05-14 11:47:54.488,1778777265244
2,1778759275566,2026-05-14 11:47:55.566,1778777266322
3,1778759276489,2026-05-14 11:47:56.489,1778777267245
4,1778759277585,2026-05-14 11:47:57.585,1778777268341


In [8]:
df_obd.to_csv("offset_fast.csv")

In [14]:
import pandas as pd

# Asegurar tipo numérico
df_obd["time_unix_ms_corr"] = pd.to_numeric(df_obd["time_unix_ms_corr"], errors="coerce")
df_imu["Time_x"] = pd.to_numeric(df_imu["Time_x"], errors="coerce")

# # Ordenar obligatoriamente para merge_asof
# df_obd = df_obd.sort_values("time_unix_ms_corr").reset_index(drop=True)
# df_imu = df_imu.sort_values("Time_y").reset_index(drop=True)

# Guardar tiempos originales para calcular desfase
df_imu_aux = df_imu.copy()
df_obd_aux = df_obd.copy()

df_imu_aux["t_imu_ms"] = df_imu_aux["Time_x"]
df_obd_aux["t_obd_ms"] = df_obd_aux["time_unix_ms_corr"]

# Tolerancia en milisegundos
tol_ms = 500   # 1.5 segundos, puedes ajustar

df_sync = pd.merge_asof(
    df_imu_aux,
    df_obd_aux[[
        "t_obd_ms",
        "Vehicle speed (km/h)",
        "time_unix_ms",
        "timestamp",
        "time_unix_ms_corr",
        "timestamp_corr"
    ]],
    left_on="t_imu_ms",
    right_on="t_obd_ms",
    direction="forward",
    tolerance=tol_ms
)

# Calcular desfase
df_sync["delay_ms"] = df_sync["t_obd_ms"] - df_sync["t_imu_ms"]
df_sync["delay_s"] = df_sync["delay_ms"] / 1000.0

# Convertir velocidad a m/s
df_sync["v_obd_kmh"] = df_sync["Vehicle speed (km/h)"]
df_sync["v_obd_mps"] = df_sync["v_obd_kmh"] / 3.6

In [15]:
print(df_sync[["t_imu_ms", "t_obd_ms", "delay_s", "v_obd_kmh"]].head(20))
print(df_sync["delay_s"].describe())

         t_imu_ms      t_obd_ms  delay_s  v_obd_kmh
0   1778777321705  1.778777e+12    0.001        0.0
1   1778777321705  1.778777e+12    0.001        0.0
2   1778777321705  1.778777e+12    0.001        0.0
3   1778777321705  1.778777e+12    0.001        0.0
4   1778777321705  1.778777e+12    0.001        0.0
5   1778777321705  1.778777e+12    0.001        0.0
6   1778777321705  1.778777e+12    0.001        0.0
7   1778777321705  1.778777e+12    0.001        0.0
8   1778777321705  1.778777e+12    0.001        0.0
9   1778777321705  1.778777e+12    0.001        0.0
10  1778777321705  1.778777e+12    0.001        0.0
11  1778777321716           NaN      NaN        NaN
12  1778777321738           NaN      NaN        NaN
13  1778777321760           NaN      NaN        NaN
14  1778777321786           NaN      NaN        NaN
15  1778777321808           NaN      NaN        NaN
16  1778777321831           NaN      NaN        NaN
17  1778777321856           NaN      NaN        NaN
18  17787773

In [19]:
df_sync.to_csv("merge_fast.csv", index=False)

In [12]:
import pandas as pd

tol_ns = 5_000  # 0.5 segundos


# Asegurar que ambos estén ordenados por tiempo
df1 = df_imu.sort_values("Time_x").reset_index(drop=True)
df2 = df_obd.sort_values("time_unix_ms_corr").reset_index(drop=True)

# Merge usando df1 como base
df_merged = pd.merge_asof(
    df1,
    df2,
    on="Time",
    direction="forward",
    tolerance=tol_ns #pd.Timedelta(seconds=2)
)

KeyError: 'Requested level (Time) does not match index name (None)'

In [20]:
import pandas as pd
import numpy as np

def assign_obd_updates_to_imu(
    df_imu,
    df_obd,
    imu_time_col="Time_y",
    obd_time_col="time_unix_ms_corr",
    speed_col="Vehicle speed (km/h)",
    tolerance_s=0.2
):
    df_i = df_imu.copy()
    df_o = df_obd.copy()

    # Asegurar formato numérico
    df_i[imu_time_col] = pd.to_numeric(df_i[imu_time_col], errors="coerce")
    df_o[obd_time_col] = pd.to_numeric(df_o[obd_time_col], errors="coerce")

    df_i = df_i.dropna(subset=[imu_time_col]).copy()
    df_o = df_o.dropna(subset=[obd_time_col, speed_col]).copy()

    # Guardar índice original del IMU
    df_i = df_i.reset_index(drop=True)
    df_i["imu_idx"] = df_i.index

    df_i["t_imu_ms"] = df_i[imu_time_col]
    df_o["t_obd_ms"] = df_o[obd_time_col]

    df_i_sorted = df_i.sort_values("t_imu_ms").reset_index(drop=True)
    df_o_sorted = df_o.sort_values("t_obd_ms").reset_index(drop=True)

    tol_ms = int(tolerance_s * 1000)

    # Para cada OBD, buscar la primera muestra IMU posterior
    obd_to_imu = pd.merge_asof(
        df_o_sorted[["t_obd_ms", speed_col]],
        df_i_sorted[["t_imu_ms", "imu_idx"]],
        left_on="t_obd_ms",
        right_on="t_imu_ms",
        direction="forward",
        tolerance=tol_ms
    )

    # Eliminar OBD que no encontró IMU dentro de la tolerancia
    obd_to_imu = obd_to_imu.dropna(subset=["imu_idx"]).copy()
    obd_to_imu["imu_idx"] = obd_to_imu["imu_idx"].astype(int)

    # Si por alguna razón varias muestras OBD caen en el mismo IMU,
    # quedarse con la más cercana
    obd_to_imu["delay_ms"] = obd_to_imu["t_imu_ms"] - obd_to_imu["t_obd_ms"]
    obd_to_imu = (
        obd_to_imu
        .sort_values("delay_ms")
        .drop_duplicates(subset=["imu_idx"], keep="first")
    )

    # Preparar columnas de update
    df_i["obd_update"] = 0
    df_i["v_obd_kmh"] = np.nan
    df_i["v_obd_mps"] = np.nan
    df_i["t_obd_ms"] = np.nan
    df_i["obd_delay_ms"] = np.nan
    df_i["obd_delay_s"] = np.nan

    # Insertar mediciones OBD solo en una fila del IMU
    idx = obd_to_imu["imu_idx"].to_numpy()

    df_i.loc[idx, "obd_update"] = 1
    df_i.loc[idx, "v_obd_kmh"] = obd_to_imu[speed_col].to_numpy()
    df_i.loc[idx, "v_obd_mps"] = obd_to_imu[speed_col].to_numpy() / 3.6
    df_i.loc[idx, "t_obd_ms"] = obd_to_imu["t_obd_ms"].to_numpy()
    df_i.loc[idx, "obd_delay_ms"] = obd_to_imu["delay_ms"].to_numpy()
    df_i.loc[idx, "obd_delay_s"] = df_i.loc[idx, "obd_delay_ms"] / 1000.0

    return df_i, obd_to_imu

In [25]:
df_imu_kf, obd_matches = assign_obd_updates_to_imu(
    df_imu=df_imu,
    df_obd=df_obd,
    imu_time_col="Time_y",
    obd_time_col="time_unix_ms_corr",
    speed_col="Vehicle speed (km/h)",
    tolerance_s=0.2
)

df_imu_kf.to_csv("merge_fast2.csv")

In [26]:
print(df_imu_kf[["t_imu_ms", "t_obd_ms", "obd_delay_s", "v_obd_kmh", "v_obd_mps", "obd_update"]]
      .query("obd_update == 1")
      .head(20))

          t_imu_ms      t_obd_ms  obd_delay_s  v_obd_kmh  v_obd_mps  \
11   1778777321716  1.778777e+12        0.010        0.0   0.000000   
43   1778777322466  1.778777e+12        0.009        0.0   0.000000   
94   1778777323657  1.778777e+12        0.010        0.0   0.000000   
139  1778777324709  1.778777e+12        0.004        0.0   0.000000   
189  1778777325879  1.778777e+12        0.021        0.0   0.000000   
243  1778777327146  1.778777e+12        0.001        0.0   0.000000   
323  1778777329017  1.778777e+12        0.022        0.0   0.000000   
369  1778777330091  1.778777e+12        0.002        0.0   0.000000   
418  1778777331238  1.778777e+12        0.017        0.0   0.000000   
450  1778777331988  1.778777e+12        0.014        0.0   0.000000   
500  1778777333158  1.778777e+12        0.017        0.0   0.000000   
532  1778777333907  1.778777e+12        0.007        0.0   0.000000   
579  1778777335007  1.778777e+12        0.007        0.0   0.000000   
629  1